In [ ]:
import os
from pathlib import Path
import numpy as np
from PIL import Image, ImageDraw

IMAGES_DIR = Path(r"D:\Harisankar\MSc. Remote Sensing and Geoinformatics\Internship2 @ DLR\new data for journal\scene_clip_building\scene_clip_building\images")      # folder with your images
NPY_DIR     = Path(r"D:\Harisankar\MSc. Remote Sensing and Geoinformatics\Internship2 @ DLR\new data for journal\scene_clip_building\scene_clip_building\npy")  # folder with .npy (same stem as images)
OUT_DIR     = Path(r"D:\Harisankar\MSc. Remote Sensing and Geoinformatics\Internship2 @ DLR\new data for journal\scene_clip_building\scene_clip_building\masks")  # output folder for masks
SAVE_OVERLAY = True                       # also save a quick overlay for QC

OUT_DIR.mkdir(parents=True, exist_ok=True)
if SAVE_OVERLAY:
    (OUT_DIR / "overlay").mkdir(parents=True, exist_ok=True)

# Which image extensions to consider
IMG_EXTS = {".png", ".jpg", ".jpeg", ".tif", ".tiff"}

def find_npy_for_image(img_path: Path) -> Path | None:
    # match by stem (e.g., "002847" ↔ "002847.npy")
    stem = img_path.stem
    cand = NPY_DIR / f"{stem}.npy"
    return cand if cand.exists() else None

def draw_polygons(mask_img: Image.Image, polys: list[np.ndarray]) -> None:
    """Fill polygons on the mask image (in-place)."""
    draw = ImageDraw.Draw(mask_img)
    for poly in polys:
        if not isinstance(poly, np.ndarray) or poly.ndim != 2 or poly.shape[1] != 2:
            continue
        # PIL expects (x, y) tuples
        pts = [tuple(map(float, p)) for p in poly]
        # Draw filled polygon as 255 (white)
        draw.polygon(pts, fill=255)

In [ ]:
def main():
    images = [p for p in IMAGES_DIR.iterdir() if p.suffix.lower() in IMG_EXTS]
    if not images:
        print(f"No images found in {IMAGES_DIR}")
        return

    for img_path in images:
        npy_path = find_npy_for_image(img_path)
        if npy_path is None:
            print(f"No .npy for: {img_path.name} (skipping)")
            continue

        # Load image to get size
        img = Image.open(img_path).convert("RGB")
        W, H = img.size

        # Prepare blank mask
        mask = Image.new("L", (W, H), 0)  # 0 background

        # Load polygons (usually an object array: [array(Nx2), array(Mx2), ...])
        polys_obj = np.load(npy_path, allow_pickle=True)

        # Normalize to a list of arrays
        if isinstance(polys_obj, np.ndarray) and polys_obj.dtype == object:
            polygons = [np.asarray(p) for p in polys_obj.tolist()]
        else:
            # Single polygon case (Nx2)
            polygons = [np.asarray(polys_obj)]

        # Draw
        draw_polygons(mask, polygons)

        # Save mask (same stem as image)
        out_mask = OUT_DIR / f"{img_path.stem}.png"
        mask.save(out_mask)

        # Optional overlay for quick visual check
        if SAVE_OVERLAY:
            overlay = img.copy()
            # outline only (optional): draw polygon edges at 2px width
            draw = ImageDraw.Draw(overlay)
            for poly in polygons:
                pts = [tuple(map(float, p)) for p in poly]
                draw.line(pts + [pts[0]], width=2, fill=(255, 0, 0))
            overlay.save(OUT_DIR / "overlay" / f"{img_path.stem}_overlay.png")

        print(f"Saved: {out_mask.name}")

if __name__ == "__main__":
    main()

Saved: 000001.png
Saved: 000002.png
Saved: 000003.png
Saved: 000004.png
Saved: 000005.png
Saved: 000006.png
Saved: 000007.png
Saved: 000008.png
Saved: 000009.png
Saved: 000010.png
Saved: 000011.png
Saved: 000012.png
Saved: 000013.png
Saved: 000014.png
Saved: 000015.png
Saved: 000016.png
Saved: 000017.png
Saved: 000018.png
Saved: 000019.png
Saved: 000020.png
Saved: 000021.png
Saved: 000022.png
Saved: 000023.png
Saved: 000024.png
Saved: 000025.png
Saved: 000026.png
Saved: 000027.png
Saved: 000028.png
Saved: 000029.png
Saved: 000030.png
Saved: 000031.png
Saved: 000032.png
Saved: 000033.png
Saved: 000034.png
Saved: 000035.png
Saved: 000036.png
Saved: 000037.png
Saved: 000038.png
Saved: 000039.png
Saved: 000040.png
Saved: 000041.png
Saved: 000042.png
Saved: 000043.png
Saved: 000044.png
Saved: 000045.png
Saved: 000046.png
Saved: 000047.png
Saved: 000048.png
Saved: 000049.png
Saved: 000050.png
Saved: 000051.png
Saved: 000052.png
Saved: 000053.png
Saved: 000054.png
Saved: 000055.png
Saved: 000